In [5]:
api_key1=''
#add key from aistudio

In [3]:
!pip install google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.5/832.5 kB 5.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 20.0 MB/s eta 0:00:00
  Attempting uninstall: anyio
    Found existing installation: anyio 4.0.0
    Uninstalling anyio-4.0.0:
      Successfully uninstalled anyio-4.0.0


In [22]:
from pypdf import PdfReader
from google import genai
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import pandas as pd
import numpy as np
import glob
import re

# -----------------------------------
# Read PDFs
# -----------------------------------
text = ""

for pdf_file in glob.glob("../resources/pdfs/*.pdf"):
    reader = PdfReader(pdf_file)

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + " "

# -----------------------------------
# Extract words
# -----------------------------------
words = re.findall(r"\b[a-zA-Z]+\b", text.lower())

# Remove very short words
words = [w for w in words if len(w) > 3]

# Take most common words only
pdf_words = [
    word
    for word, count in Counter(words).most_common(500)
]

print("Candidate words:", len(pdf_words))

# -----------------------------------
# Search words
# -----------------------------------
word_to_find = [
    "Bharath",
    "Deepankar",
    "Hari",
    "Abani"
]

# -----------------------------------
# Gemini Client
# -----------------------------------
client = genai.Client(api_key=api_key1)

# -----------------------------------
# Batch embedding helper
# -----------------------------------
def get_embeddings(items):

    embeddings = []

    for i in range(0, len(items), 100):

        batch = items[i:i+100]

        result = client.models.embed_content(
            model="gemini-embedding-001",
            contents=batch
        )

        embeddings.extend(
            [e.values for e in result.embeddings]
        )

    return np.array(embeddings)

# -----------------------------------
# Create embeddings
# -----------------------------------
search_embeddings = get_embeddings(word_to_find)

pdf_embeddings = get_embeddings(pdf_words)

# -----------------------------------
# Find top 5 matching words
# -----------------------------------
rows = []

for idx, search_word in enumerate(word_to_find):

    similarities = cosine_similarity(
        [search_embeddings[idx]],
        pdf_embeddings
    )[0]

    top_indices = similarities.argsort()[-5:][::-1]

    matches = [
        pdf_words[i]
        for i in top_indices
    ]

    rows.append([search_word] + matches)

# -----------------------------------
# DataFrame
# -----------------------------------
df = pd.DataFrame(
    rows,
    columns=[
        "word",
        "match1",
        "match2",
        "match3",
        "match4",
        "match5"
    ]
)

print(df)

Candidate words: 326


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 21.01448448s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-